# microWakeWord Training Notebook — "hey frank"

Tuned for:
- **Wake word:** `hey frank` (phonetic: `"hˈeɪ fɹˈæŋk˺"` — use `--phoneme-input` for consistent hard K)
- **Hardware:** Local NVIDIA GPU via WSL2 (tested on 12 GB VRAM)
- **Target:** M5Stack devices via ESPHome

Based on the official `basic_training_notebook.ipynb` with practical improvements.
**Run in WSL2 with the wakeword-env venv. Use Python 3.12.**

## Workflow

### Run on Google Colab (requires piper-tts, Linux only):
| Cell | Description |
|---|---|
| 2 | Install deps, generate 1 preview sample, verify pronunciation |
| 3 | Generate full positive sample set (50k recommended) |
| 4 | Generate confusable negative samples (hey fran, hey finn, etc.) |

### Run locally in WSL2:
| Cell | Description |
|---|---|
| 1 | Install microWakeWord — **restart kernel after** |
| 5 | Download augmentation audio (RIRs, AudioSet, FMA) |
| 6 | Download pre-built negative datasets |
| 7 | Configure augmentation pipeline |
| 8 | Generate spectrogram features for positive samples |
| 9 | Preview an augmented positive clip |
| 10 | Generate spectrogram features for confusable negatives |
| 11 | Preview an augmented confusable clip |
| 12 | Write training_parameters.yaml |
| 13 | Train (run via tmux for long runs) |
| 14 | Export .tflite + print ESPHome manifest |

## Quick-edit constants:
| Setting | Value | Notes |
|---|---|---|
| `target_word` | `"hˈeɪ fɹˈæŋk˺"` | IPA phoneme input — forces hard K |
| `MAX_SAMPLES` | `50_000` | Reduce to 25k if storage is tight |
| `PIPER_BATCH` | `256` | Reduce to 128 if CUDA OOM |
| `CONFUSABLE_SAMPLES_PER_PHRASE` | `2000` | ~26k total across 13 phrases |
| `batch_size` (YAML) | `256` | Reduce to 128 if OOM during training |
| Training steps | `[20000, 15000, 10000]` | 45k total; use `[5000]` for a quick test |
| `target_minimization` | `0.3` FA/hr | Lower = stricter false accept control |

**Upstream warning:** The model may not be usable without experimentation. Treat this as a starting point!


In [ ]:
# ── Install microWakeWord ────────────────────────────────────────────────────
# RESTART the kernel after this cell finishes!
# On re-runs: microWakeWord/ clone is skipped automatically if it already exists.

import platform, os

if platform.system() == "Darwin":
    # macOS / Apple Silicon only — not needed for Colab/NVIDIA
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# audio-metadata fork unpins `attrs` so Jupyter doesn't break
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

if not os.path.exists('microWakeWord'):
    !git clone https://github.com/kahrendt/microWakeWord
else:
    print("microWakeWord already cloned, skipping")

!pip install -e ./microWakeWord

print("\n✅ Done. RESTART YOUR KERNEL before running any other cells!")

In [ ]:
# -- Wake word config + single preview sample --------------------------------
#
# target_word: IPA phoneme string for "hey frank".
#   hˈeɪ fɹˈæŋk˺  forces a hard final K via the closing tie-bar (˺).
#   If the preview sounds off, try removing ˺ or adjusting vowels.
#   Always use --phoneme-input with IPA strings.
#
# MAX_SAMPLES: 50k is the sweet spot for GPU training.
#   50_000 = best quality   25_000 = good if disk space is tight
#
# PIPER_BATCH: clips per GPU pass. Reduce to 128 if CUDA OOM.

# RUN IN GOOGLE COLAB (or WSL2 -- piper requires Linux)

target_word = "hˈeɪ fɹˈæŋk˺"  # <-- EDIT: IPA phoneme input
MAX_SAMPLES = 50_000            # <-- EDIT: 50k recommended
PIPER_BATCH = 256               # <-- EDIT: reduce to 128 if CUDA OOM

import glob, os, shutil, subprocess, sys, urllib.request
from IPython.display import Audio

MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
MODEL_CONFIG_PATH = f'{MODEL_PATH}.json'
PIPER_REPO_DIR = '/content/piper'
PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
PIPER_PYTHON_DIR = f'{PIPER_REPO_DIR}/src/python'
MONOTONIC_ALIGN_DIR = f'{PIPER_PYTHON_DIR}/piper_train/vits/monotonic_align'
MONOTONIC_ALIGN_IMPORT_DIR = f'{MONOTONIC_ALIGN_DIR}/monotonic_align'
MONOTONIC_ALIGN_BUILD_DIR = f'{MONOTONIC_ALIGN_DIR}/piper_train/vits/monotonic_align'

# -- Install current piper-sample-generator deps for Colab --------------------
# Keep the same generator model/output behavior, but avoid installing Piper's
# full training package. We only need piper-tts plus Piper's source tree for
# piper_train, with monotonic_align compiled in-place.
!apt-get -qq update
!apt-get -qq install -y espeak-ng
!{sys.executable} -m pip install -q --upgrade pip setuptools wheel cython
!{sys.executable} -m pip install -q --upgrade piper-tts piper-sample-generator

if not os.path.exists(PIPER_REPO_DIR):
    !git clone --depth 1 https://github.com/rhasspy/piper {PIPER_REPO_DIR}
else:
    print('piper repo already present, skipping clone')

if not os.path.exists(PIPER_SAMPLE_GENERATOR_DIR):
    !git clone --depth 1 https://github.com/rhasspy/piper-sample-generator {PIPER_SAMPLE_GENERATOR_DIR}
else:
    print('piper-sample-generator repo already present, skipping clone')

# Piper imports:
# piper_train.vits.monotonic_align.monotonic_align.core
# Its setup.py emits the extension into a relative piper_train/... path, so we
# stage that path, build, then copy the resulting binary into the import dir.
shutil.rmtree(f'{PIPER_PYTHON_DIR}/build', ignore_errors=True)
shutil.rmtree(MONOTONIC_ALIGN_IMPORT_DIR, ignore_errors=True)
shutil.rmtree(f'{MONOTONIC_ALIGN_DIR}/piper_train', ignore_errors=True)
os.makedirs(MONOTONIC_ALIGN_IMPORT_DIR, exist_ok=True)
os.makedirs(MONOTONIC_ALIGN_BUILD_DIR, exist_ok=True)
open(f'{MONOTONIC_ALIGN_IMPORT_DIR}/__init__.py', 'a').close()
!cd {MONOTONIC_ALIGN_DIR} && {sys.executable} setup.py build_ext --inplace

built_ext = next(iter(glob.glob(f'{MONOTONIC_ALIGN_BUILD_DIR}/core.*')), None)
if not built_ext:
    raise FileNotFoundError('monotonic_align build did not produce core extension')
shutil.copy2(built_ext, f'{MONOTONIC_ALIGN_IMPORT_DIR}/{os.path.basename(built_ext)}')

# -- Download LibriTTS-R generator model + config -----------------------------
model_url = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
model_config_url = 'https://raw.githubusercontent.com/rhasspy/piper-sample-generator/master/models/en_US-libritts_r-medium.pt.json'
repo_model_config_path = f'{PIPER_SAMPLE_GENERATOR_DIR}/models/en_US-libritts_r-medium.pt.json'
os.makedirs('models', exist_ok=True)
if not os.path.exists(MODEL_PATH):
    print('Downloading TTS model (~300 MB)...')
    urllib.request.urlretrieve(model_url, MODEL_PATH)
    print('Model downloaded')
else:
    print('Model already present')

if not os.path.exists(MODEL_CONFIG_PATH):
    if os.path.exists(repo_model_config_path):
        print('Copying model config from cloned repo...')
        shutil.copy2(repo_model_config_path, MODEL_CONFIG_PATH)
    else:
        print('Downloading model config...')
        urllib.request.urlretrieve(model_config_url, MODEL_CONFIG_PATH)
    print('Model config downloaded')
else:
    print('Model config already present')

# -- Shared runtime helper ----------------------------------------------------
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([
    PIPER_PYTHON_DIR,
    PIPER_SAMPLE_GENERATOR_DIR,
    env.get('PYTHONPATH', ''),
])
os.environ['PYTHONPATH'] = env['PYTHONPATH']

# Surface the real Piper error in Colab instead of only showing exit code 1.
def run_piper_generator(args):
    result = subprocess.run(
        [sys.executable, '-m', 'piper_sample_generator', *args],
        env=os.environ.copy(),
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

# -- Generate 1 sample to verify pronunciation --------------------------------
run_piper_generator([
    target_word,
    '--phoneme-input',
    '--model', MODEL_PATH,
    '--max-samples', '1',
    '--batch-size', '1',
    '--noise-scales', '0.5',
    '--noise-scale-ws', '0.6',
    '--output-dir', 'generated_samples',
])

print()
print('Listen below -- should sound like "hey frank" with a clear hard K.')
print('If not, edit target_word above and re-run before proceeding.')
Audio(filename='generated_samples/0.wav', autoplay=True)


In [ ]:
# -- Generate full training sample set ---------------------------------------
# Generates MAX_SAMPLES TTS clips using the LibriTTS-R multi-speaker model
# (904 speakers with SLERP voice blending = high variety).
#
# At PIPER_BATCH=256 on a Colab T4 GPU this takes roughly:
#   10k samples ~  5 min
#   25k samples ~ 12 min
#   50k samples ~ 25 min
#
# Noise/length parameters:
#   --noise-scales     phoneme-level variation (lower = more consistent K sound)
#   --noise-scale-ws   rhythm/duration variation
#   --length-scales    speaking rate (<1 = faster, >1 = slower)

# RUN IN GOOGLE COLAB (or WSL2)

run_piper_generator([
    target_word,
    '--phoneme-input',
    '--model', MODEL_PATH,
    '--max-samples', str(MAX_SAMPLES),
    '--batch-size', str(PIPER_BATCH),
    '--noise-scales', '0.5',
    '--noise-scale-ws', '0.6',
    '--output-dir', 'generated_samples',
])

n = len([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
print(f'\n {n} samples in ./generated_samples/')


In [ ]:
# -- Generate confusable negative samples ------------------------------------
# Phonetically-similar phrases that must NOT trigger "hey frank".
# These become high-penalty hard negatives during training, teaching the model
# to discriminate fine-grained phonetic differences like fr- vs fl-, fn-, etc.
#
# Uses plain text (no --phoneme-input) so piper generates natural variation
# in how these similar-sounding words are pronounced.
#
# RUN IN GOOGLE COLAB or WSL2 (requires piper-sample-generator, Linux only).
#
# CONFUSABLE_SAMPLES_PER_PHRASE:
#   500  = fast/light (~6500 total)
#   2000 = recommended (~26k total)
#   5000 = maximum quality (~65k total)

CONFUSABLE_SAMPLES_PER_PHRASE = 2000  # <-- EDIT

confusable_phrases = [
    'hey frin',       # drops the a before n
    'hey fran',       # very similar, missing final k
    'hey flin',       # swaps r for l
    'hey finn',       # drops the r entirely
    'hey friend',     # common phrase, similar fr- onset
    'hey fringe',     # adds final zh sound
    'hey fred',       # same fr- onset, different vowel
    'hey frog',       # same fr- onset
    'hey france',     # france = /fraens/, very close
    'hey franklin',   # frank as a prefix
    'frank',          # missing the hey prefix
    'hey blank',      # replaces fr- with bl-
    'hey franz',      # German name, very close phonetically
]

import os, shutil, subprocess, sys
from pathlib import Path

os.makedirs('confusable_negatives', exist_ok=True)

for phrase in confusable_phrases:
    safe_name = phrase.replace(' ', '_')
    tmp_dir = f'/tmp/confusable_{safe_name}'

    existing = len(list(Path('confusable_negatives').glob(f'{safe_name}_*.wav')))
    if existing >= CONFUSABLE_SAMPLES_PER_PHRASE:
        print(f'  "{phrase}": {existing} samples already present, skipping')
        continue

    print(f'  "{phrase}" -> {CONFUSABLE_SAMPLES_PER_PHRASE} samples...')
    os.makedirs(tmp_dir, exist_ok=True)

    # subprocess avoids {var} interpolation issue with ! inside loops
    run_piper_generator([
        phrase,
        '--model', MODEL_PATH,
        '--max-samples', str(CONFUSABLE_SAMPLES_PER_PHRASE),
        '--batch-size', str(PIPER_BATCH),
        '--output-dir', tmp_dir,
    ])

    copied = 0
    for wav in sorted(Path(tmp_dir).glob('*.wav')):
        dest = Path('confusable_negatives') / f'{safe_name}_{wav.name}'
        shutil.copy(wav, dest)
        copied += 1
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f'    {copied} samples -> confusable_negatives/')

total = len(list(Path('confusable_negatives').glob('*.wav')))
print(f'\n {total} total confusable negative samples in ./confusable_negatives/')


In [ ]:
# ── Download augmentation audio ───────────────────────────────────────────────
# Downloads room impulse responses (reverb simulation) and background noise.
# Uses decode=False + soundfile to avoid torchcodec API incompatibilities.
#
# NOTE: Mixed licenses — suitable for non-commercial personal use only.

import datasets, scipy, os, sys, io, urllib.request, zipfile, numpy as np
import librosa, soundfile as sf, fsspec
from pathlib import Path
from tqdm import tqdm

def _decode_audio(audio_info, target_sr=16000):
    """Decode audio from a datasets Audio(decode=False) row using soundfile.
    Handles both inline bytes and hf:// path references."""
    data = audio_info.get('bytes')
    if not data:
        path = audio_info.get('path') or ''
        if path.startswith('hf://'):
            with fsspec.open(path, 'rb') as f:
                data = f.read()
        else:
            data = open(path, 'rb').read()
    arr, sr = sf.read(io.BytesIO(data), dtype='float32', always_2d=False)
    if arr.ndim > 1:
        arr = arr.mean(axis=1)  # stereo -> mono
    if sr != target_sr:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=target_sr)
    return arr

# ── MIT Room Impulse Responses ───────────────────────────────────────────────
rir_wavs = list(Path('mit_rirs').glob('*.wav')) if Path('mit_rirs').exists() else []
if len(rir_wavs) < 10:
    print(f'Downloading MIT RIRs (found {len(rir_wavs)} existing)...')
    os.makedirs('./mit_rirs', exist_ok=True)
    rir_ds = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True
    )
    rir_ds = rir_ds.cast_column('audio', datasets.Audio(decode=False))
    for i, row in enumerate(tqdm(rir_ds)):
        # path may be hf:// URL or None — use index as fallback filename
        raw_name = (row['audio'].get('path') or '').split('/')[-1].split('\\')[-1]
        name = raw_name if raw_name.lower().endswith('.wav') else f'rir_{i:04d}.wav'
        try:
            arr = _decode_audio(row['audio'])
            scipy.io.wavfile.write(f"mit_rirs/{name}", 16000, (arr * 32767).astype(np.int16))
        except Exception as e:
            print(f'  ⚠️  Skipped row {i}: {e}')
    n = len(list(Path('mit_rirs').glob('*.wav')))
    print(f'✅ MIT RIRs done ({n} files)')
else:
    print(f'✅ MIT RIRs already present ({len(rir_wavs)} files)')

# ── AudioSet (~2000 clips, streaming — old tar files removed Sep 2025) ────────
AUDIOSET_CLIPS = 18683
if not os.path.exists('audioset_16k') or len(list(Path('audioset_16k').glob('*.wav'))) < 100:
    print(f'Downloading AudioSet ({AUDIOSET_CLIPS} clips)...')
    os.makedirs('audioset_16k', exist_ok=True)
    ds = datasets.load_dataset(
        'agkphysics/AudioSet', 'balanced', split='train', streaming=True, trust_remote_code=True
    )
    ds = ds.cast_column('audio', datasets.Audio(decode=False))
    for i, row in enumerate(tqdm(ds, total=AUDIOSET_CLIPS)):
        if i >= AUDIOSET_CLIPS:
            break
        try:
            arr = _decode_audio(row['audio'])
            scipy.io.wavfile.write(f"audioset_16k/{i:05d}.wav", 16000, (arr * 32767).astype(np.int16))
        except Exception:
            pass
    print('✅ AudioSet done')
else:
    print('✅ AudioSet already present')

# ── Free Music Archive (xsmall) ───────────────────────────────────────────────
if not os.path.exists('fma'):
    print('Downloading FMA xsmall...')
    os.makedirs('fma', exist_ok=True)
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip',
        'fma/fma_xs.zip'
    )
    with zipfile.ZipFile('fma/fma_xs.zip') as zf:
        zf.extractall('fma')

if not os.path.exists('fma_16k') or len(list(Path('fma_16k').glob('*.wav'))) < 100:
    print('Converting FMA to 16kHz WAV...')
    os.makedirs('fma_16k', exist_ok=True)
    for p in tqdm(sorted(Path('fma').glob('**/*.mp3'))):
        try:
            arr, _ = librosa.load(str(p), sr=16000, mono=True)
            scipy.io.wavfile.write(f"fma_16k/{p.stem}.wav", 16000, (arr * 32767).astype(np.int16))
        except Exception:
            pass
    print('✅ FMA done')
else:
    print('✅ FMA already present')

In [ ]:
# ── Download pre-generated negative datasets ──────────────────────────────────
# Real-world ambient/speech recordings pre-processed by the microWakeWord project.
#
#  speech            — general speech (hardest negatives for 'hey amy')
#  dinner_party      — multi-speaker conversation + background noise
#  no_speech         — ambient/music with no speech
#  dinner_party_eval — held-out eval set for ambient false-positives-per-hour metric

import os, urllib.request, zipfile

if not os.path.exists('./negative_datasets'):
    print('Downloading negative datasets...')
    os.makedirs('./negative_datasets', exist_ok=True)
    link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
    for fname in ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']:
        zip_path = f'negative_datasets/{fname}'
        urllib.request.urlretrieve(link_root + fname, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall('negative_datasets')
        print(f'  ✅ {fname}')
    print('✅ All negative datasets ready')
else:
    print('✅ Negative datasets already present')

In [ ]:
# ── Configure augmentation pipeline ──────────────────────────────────────────
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from pathlib import Path
import os

# ── Verify data directories ───────────────────────────────────────────────────
print(f'Working directory: {os.getcwd()}')
for d in ['mit_rirs', 'fma_16k', 'audioset_16k', 'generated_samples']:
    p = Path(d)
    if p.exists():
        n = len(list(p.glob('*.wav')))
        print(f'  ✅ {d}: {n} .wav files  ({p.resolve()})')
    else:
        print(f'  ❌ {d}: NOT FOUND  (expected at {p.resolve()})')

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15,
        'TanhDistortion':        0.10,
        'PitchShift':            0.15,
        'BandStopFilter':        0.10,
        'AddColorNoise':         0.20,
        'AddBackgroundNoise':    0.85,
        'Gain':                  1.00,
        'GainTransition':        0.25,
        'RIR':                   0.60,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5,
    background_max_snr_db=20,
    min_jitter_s=0.10,
    max_jitter_s=0.50,
)

print('✅ Augmentation pipeline ready')

In [ ]:
# ── Generate spectrogram features ─────────────────────────────────────────────
# Converts augmented clips into 40-band spectrogram features (same format the
# M5Stack micro_speech preprocessor produces at runtime).
#
# training:   slide_frames=10, repeat=3 — each clip generates 3 augmented versions
# validation: slide_frames=10, repeat=1
# testing:    slide_frames=1,  repeat=1 — simulates actual streaming inference
#
# At 50k samples on an NVIDIA GPU this takes roughly 20–40 minutes.
# Already-generated splits are skipped automatically on re-runs.

import os, shutil, traceback
from mmap_ninja.ragged import RaggedMmap

os.makedirs('generated_augmented_features', exist_ok=True)

split_config = {
    'training':   {'split_name': 'train',      'repetition': 3, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
}

for split, cfg in split_config.items():
    out_dir   = f'generated_augmented_features/{split}'
    mmap_path = f'{out_dir}/wakeword_mmap'

    # Check for valid (non-empty) existing mmap
    if os.path.exists(mmap_path):
        mmap_files = list(os.scandir(mmap_path))
        if mmap_files:
            print(f'⏭  {split}: already exists ({len(mmap_files)} files), skipping')
            continue
        else:
            print(f'⚠️  {split}: mmap dir exists but is empty — removing and regenerating')
            shutil.rmtree(mmap_path)

    os.makedirs(out_dir, exist_ok=True)
    print(f'\n⚙️  Generating {split} (slide_frames={cfg["slide_frames"]}, repeat={cfg["repetition"]})...')

    try:
        spectrograms = SpectrogramGeneration(
            clips=clips,
            augmenter=augmenter,
            slide_frames=cfg['slide_frames'],
            step_ms=10,
        )
        RaggedMmap.from_generator(
            out_dir=mmap_path,
            sample_generator=spectrograms.spectrogram_generator(
                split=cfg['split_name'],
                repeat=cfg['repetition'],
            ),
            batch_size=200,
            verbose=True,
        )
        print(f'✅ {split} done')
    except Exception:
        print(f'\n❌ {split} FAILED — full traceback:')
        traceback.print_exc()
        # Remove partial mmap so next run retries cleanly
        if os.path.exists(mmap_path):
            shutil.rmtree(mmap_path)
            print(f'   Removed partial mmap at {mmap_path}')
        break

print('\n✅ All feature sets ready')

In [ ]:
# -- Preview an augmented sample ---------------------------------------------
# Run this a few times. You should still be able to hear 'hey frank' even
# through heavy noise/reverb. If it is completely unintelligible, reduce the
# SNR range (raise background_min_snr_db toward 0) or lower the RIR probability.

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip    = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

print('Augmented preview (re-run for variety):')
Audio(filename='augmented_clip.wav', autoplay=True)


In [ ]:
# ── Generate spectrogram features for confusable negatives ───────────────────
# Augments the confusable near-miss clips and converts them to spectrogram mmaps,
# using the same pipeline as the positive samples (cell 7).
#
# RUN LOCALLY alongside cells 4–11 (after confusable_negatives/ is synced down).
#
# This cell is idempotent — already-generated splits are skipped automatically.

import os, shutil, traceback
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
from pathlib import Path

n_confusable = len(list(Path('confusable_negatives').glob('*.wav')))
if n_confusable == 0:
    print('❌ confusable_negatives/ is empty — run the confusable generation cell first.')
else:
    print(f'✅ Found {n_confusable} confusable negative samples')

    confusable_clips = Clips(
        input_directory='confusable_negatives',
        file_pattern='*.wav',
        max_clip_duration_s=None,
        remove_silence=True,
        random_split_seed=99,    # different seed from the positive clips
        split_count=0.1,
    )

    os.makedirs('confusable_features', exist_ok=True)

    split_config = {
        'training':   {'split_name': 'train',      'repetition': 2, 'slide_frames': 10},
        'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
        'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
    }

    for split, cfg in split_config.items():
        out_dir   = f'confusable_features/{split}'
        mmap_path = f'{out_dir}/wakeword_mmap'

        if os.path.exists(mmap_path) and list(os.scandir(mmap_path)):
            print(f'⏭  {split}: already exists, skipping')
            continue

        if os.path.exists(mmap_path):
            shutil.rmtree(mmap_path)
        os.makedirs(out_dir, exist_ok=True)

        print(f'\n⚙️  Generating confusable {split} features...')
        try:
            spectrograms = SpectrogramGeneration(
                clips=confusable_clips,
                augmenter=augmenter,
                slide_frames=cfg['slide_frames'],
                step_ms=10,
            )
            RaggedMmap.from_generator(
                out_dir=mmap_path,
                sample_generator=spectrograms.spectrogram_generator(
                    split=cfg['split_name'],
                    repeat=cfg['repetition'],
                ),
                batch_size=200,
                verbose=True,
            )
            print(f'✅ {split} done')
        except Exception:
            print(f'\n❌ {split} FAILED:')
            traceback.print_exc()
            if os.path.exists(mmap_path):
                shutil.rmtree(mmap_path)
            break

    print('\n✅ Confusable features ready')


In [ ]:
# ── Preview an augmented confusable sample ───────────────────────────────────
# Run this a few times to spot-check the confusable negatives.
# You should hear something like "hey fran" or "hey finn" — clearly NOT "hey frank".
# If a clip sounds identical to "hey frank", that phrase may need rephrasing.

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_confusable = confusable_clips.get_random_clip()
augmented_confusable = augmenter.augment_clip(random_confusable)
save_clip(augmented_confusable, 'augmented_confusable.wav')

print('Augmented confusable preview (re-run for variety):')
Audio(filename='augmented_confusable.wav', autoplay=True)


In [1]:
# -- Write training configuration YAML ----------------------------------------
# Tuned for 'hey frank' on a local NVIDIA GPU (12 GB VRAM).
#
# BEFORE RUNNING: make sure you have run cells 7 -> 10 first so that
# generated_augmented_features/ and confusable_features/ both exist.
# If confusable_features/ is missing, set SKIP_CONFUSABLES = True below.
#
# Batch composition (approximate share of each training batch):
#   positives       6.0  -> 13%
#   speech         10.0  -> 21%
#   dinner_party   15.0  -> 32%   (boosted -- covers TV/ambient speech)
#   no_speech       5.0  -> 11%
#   confusables     8.0  -> 17%
#
# Key knobs:
#   negative_class_weight -- global false-accept penalty; raise if still triggering on ambient
#   dinner_party penalty_weight -- targeted penalty for TV/conversation-style audio
#   confusable penalty_weight -- targeted penalty for near-miss phrases
#   target_minimization -- FA/hr threshold trainer must beat before maximizing recall

import yaml, os
from pathlib import Path

SKIP_CONFUSABLES = not Path('confusable_features/training/wakeword_mmap').exists()
if SKIP_CONFUSABLES:
    print('WARNING: confusable_features/ not found -- training WITHOUT confusable negatives.')
    print('   Run cells 7 -> 10 first, then re-run this cell for best results.')
else:
    print('confusable_features/ found -- confusable negatives included.')

config = {}
config['window_step_ms'] = 10
config['train_dir']      = 'trained_models/hey_frank'

config['features'] = [
    {
        'features_dir':         'generated_augmented_features',
        'sampling_weight':      8.0,
        'penalty_weight':       2.0,
        'truth':                True,
        'truncation_strategy':  'truncate_start',
        'type':                 'mmap',
    },
    {
        'features_dir':         'negative_datasets/speech',
        'sampling_weight':      10.0,
        'penalty_weight':       2.5,
        'truth':                False,
        'truncation_strategy':  'random',
        'type':                 'mmap',
    },
    {
        # Multi-speaker + background noise -- primary source of ambient/TV false accepts
        'features_dir':         'negative_datasets/dinner_party',
        'sampling_weight':      15.0,
        'penalty_weight':       3.0,
        'truth':                False,
        'truncation_strategy':  'random',
        'type':                 'mmap',
    },
    {
        'features_dir':         'negative_datasets/no_speech',
        'sampling_weight':      5.0,
        'penalty_weight':       1.0,
        'truth':                False,
        'truncation_strategy':  'random',
        'type':                 'mmap',
    },
    {
        # Eval-only -- sampling_weight=0 means never in training batches
        'features_dir':         'negative_datasets/dinner_party_eval',
        'sampling_weight':      0.0,
        'penalty_weight':       1.0,
        'truth':                False,
        'truncation_strategy':  'split',
        'type':                 'mmap',
    },
]

if not SKIP_CONFUSABLES:
    config['features'].append({
        'features_dir':         'confusable_features',
        'sampling_weight':      8.0,
        'penalty_weight':       5.0,
        'truth':                False,
        'truncation_strategy':  'random',
        'type':                 'mmap',
    })

# Two-phase training schedule
config['training_steps']        = [25000, 20000]   # <-- EDIT: [5000] for quick test
config['positive_class_weight'] = [2,     2    ]
config['negative_class_weight'] = [50,    60   ]
config['learning_rates']        = [0.001, 0.0001]
config['batch_size']            = 256

# SpecAugment
config['time_mask_max_size'] = [5, 5]
config['time_mask_count']    = [1, 1]
config['freq_mask_max_size'] = [3, 3]
config['freq_mask_count']    = [1, 1]

config['eval_step_interval'] = 500
config['clip_duration_ms']   = 1500

config['target_minimization'] = 0.4
config['minimization_metric'] = 'ambient_false_positives_per_hour'
config['maximization_metric'] = 'average_viable_recall'

os.makedirs('trained_models/hey_frank', exist_ok=True)
with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)

n_features = len(config['features'])
total_weight = sum(f['sampling_weight'] for f in config['features'])
print(f'training_parameters.yaml written')
print(f'   Feature sets: {n_features}  |  Phases: {len(config["training_steps"])}  |  Total steps: {sum(config["training_steps"])}')
print(f'   negative_class_weight: {config["negative_class_weight"]}  |  target FA/hr: {config["target_minimization"]}')
print(' Batch composition:')
for f in config['features']:
    pct = 100 * f['sampling_weight'] / total_weight if total_weight else 0
    label = 'POSITIVE' if f['truth'] else f'neg pw={f["penalty_weight"]}'
    print(f'     {f["features_dir"].split("/")[-1]:30s} {pct:4.0f}%  [{label}]')


confusable_features/ found -- confusable negatives included.
training_parameters.yaml written
   Feature sets: 6  |  Phases: 2  |  Total steps: 45000
   negative_class_weight: [50, 60]  |  target FA/hr: 0.4
 Batch composition:
     generated_augmented_features     17%  [POSITIVE]
     speech                           22%  [neg pw=2.5]
     dinner_party                     33%  [neg pw=3.0]
     no_speech                        11%  [neg pw=1.0]
     dinner_party_eval                 0%  [neg pw=1.0]
     confusable_features              17%  [neg pw=5.0]


In [ ]:
# ── Train the model ───────────────────────────────────────────────────────────
# NOTE: Run this cell via the WSL CLI for long training runs — it's more stable
# than keeping Jupyter open for hours. See notebook header for instructions.
#
# Use --train 0 to skip training and only export/evaluate a saved checkpoint.
# Use --restore_checkpoint 0 for a FRESH start (no existing checkpoint).
# Use --restore_checkpoint 1 to RESUME from a previous checkpoint.

# use tmux new -s frank
# or tmux attach -t frank
# then:

'''
export XLA_FLAGS='--xla_gpu_autotune_level=0'
export PATH="/home/malonestar/wakeword-env/lib/python3.12/site-packages/nvidia/cuda_nvcc/bin:$PATH"

nohup python3 -m microwakeword.model_train_eval \
  --training_config "training_parameters.yaml" \
  --train 1 \
  --restore_checkpoint 0 \
  --test_tf_nonstreaming 0 \
  --test_tflite_nonstreaming 0 \
  --test_tflite_nonstreaming_quantized 0 \
  --test_tflite_streaming 0 \
  --test_tflite_streaming_quantized 1 \
  --use_weights "best_weights" \
  mixednet \
  --pointwise_filters "64,64,64,64" \
  --repeat_in_block "1, 1, 1, 1" \
  --mixconv_kernel_sizes "[5], [7,11], [9,15], [23]" \
  --residual_connection "0,0,0,0" \
  --first_conv_filters 32 \
  --first_conv_kernel_size 5 \
  --stride 3 \
  > training.log 2>&1 &

tail -f training.log

'''

import sys, os

# Use pip-bundled ptxas 12.9 instead of system ptxas 12.4
# (required for RTX 5000 series / Blackwell CC 12.0)
_ptxas_dir = os.path.join(os.path.dirname(sys.executable),
             '../lib/python3.12/site-packages/nvidia/cuda_nvcc/bin')
_ptxas_dir = os.path.normpath(_ptxas_dir)
os.environ['PATH'] = _ptxas_dir + ':' + os.environ.get('PATH', '')
os.environ['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'

# Ensure TensorBoard is present (required by tf.summary)
!"{sys.executable}" -m pip install -q tensorboard

!"{sys.executable}" -m microwakeword.model_train_eval \
--training_config "training_parameters.yaml" \
--train 1 \
--restore_checkpoint 0 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes "[5], [7,11], [9,15], [23]" \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

print('\n✅ Training complete!')

In [ ]:
# ── Locate the trained model + print ESPHome manifest template ───────────────
# The .tflite is your deployable model.
# You need a companion JSON manifest to use it in ESPHome / M5Stack.
# A starter template is printed below — fill in the probability_cutoff
# based on the test metrics printed by the training cell above.
#
# See full docs: https://esphome.io/components/micro_wake_word
# See examples:  https://github.com/esphome/micro-wake-word-models/tree/main/models/v2

import os, json

tflite_path = 'trained_models/hey_frank/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'

if os.path.exists(tflite_path):
    size_kb = os.path.getsize(tflite_path) / 1024
    print(f'✅ Model: {tflite_path}')
    print(f'   Size:  {size_kb:.1f} KB')

    manifest = {
        'type': 'micro_wake_word_model',
        'wake_word': 'hey frank',
        'author': 'your-name',
        'website': '',
        'model': 'hey_frank.tflite',
        'version': 1,
        'micro': {
            # Adjust based on test metrics — lower = more sensitive / more false accepts
            'probability_cutoff':    0.5,   # <-- EDIT based on training output
            'sliding_window_average': 10,
            'tensor_arena_size':     30000,
        }
    }

    print('\n── ESPHome manifest template (hey_frank.json) ──')
    print(json.dumps(manifest, indent=2))

    with open('hey_frank.json', 'w') as f:
        json.dump(manifest, f, indent=2)
    print('\n✅ hey_frank.json written to working directory')
    print(f'   Copy hey_frank.tflite and hey_frank.json to your ESPHome config directory.')
else:
    print('❌ Model not found — check training output above for errors.')